In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle
from pathlib import Path
import os

df = pd.read_csv("../data/cleaned.csv")
df['posting_date'] = pd.to_datetime(df['posting_date'], errors='coerce')
df['due_in_date'] = pd.to_datetime(df['due_in_date'], errors='coerce')
df['clear_date'] = pd.to_datetime(df['clear_date'], errors='coerce')

df['posting_month'] = df['posting_date'].dt.month.fillna(6)
df['delay_days'] = (df['clear_date'] - df['due_in_date']).dt.days
df['is_q_end'] = df['posting_month'].isin([3,6,9,12]).astype(int)
df['buisness_year'] = df['buisness_year'].fillna(2020)
df['total_open_amount'] = df['total_open_amount'].fillna(df['total_open_amount'].median())
# Delay ko pehle median se bharo taaki wo bhi kaam kare
df['delay_days'] = df['delay_days'].fillna(df['delay_days'].median())

# FINAL - Median threshold = 50% data LOW, 50% HIGH
amount_threshold = df['total_open_amount'].median()
delay_threshold = 15
print(f"Amount median: {amount_threshold}, Delay: {delay_threshold}")

# Risk = Bada amount OR zyada delay
df['is_delayed'] = (
    (df['total_open_amount'] > amount_threshold) |
    (df['delay_days'] > delay_threshold)
).astype(int)

print(df['is_delayed'].value_counts(normalize=True))

features = ['total_open_amount','posting_month','delay_days','buisness_year','is_q_end']
X = df[features]
y = df['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=30, max_depth=10, random_state=42)
model.fit(X_train, y_train)

print(f"Acc: {model.score(X_test, y_test):.3f}")
# 3 tests
for amt, dly in [(6000,0),(17523,0),(50000,20)]:
    td = pd.DataFrame([{'total_open_amount':amt,'posting_month':6,'delay_days':dly,'buisness_year':2023,'is_q_end':0}])
    print(f"Amt {amt}, Delay {dly} -> {model.predict_proba(td)[0][1]*100:.1f}%")

Path("../models").mkdir(exist_ok=True)
with open("../models/model.pkl","wb") as f:
    pickle.dump(model,f,protocol=4)
with open("../models/risk_model.pkl","wb") as f:
    pickle.dump(model,f,protocol=4)

Amount median: 17559.64, Delay: 15
is_delayed
0    0.50001
1    0.49999
Name: proportion, dtype: float64
Acc: 1.000
Amt 6000, Delay 0 -> 0.0%
Amt 17523, Delay 0 -> 0.0%
Amt 50000, Delay 20 -> 100.0%
